# Evaluate a campaign – overview, equivalent-source method, report tables

Works only with the **result files** of the single evaluations (`evaluate_measurement.ipynb`, section 7) – the raw
data are not read again. Reference copy: `templates/evaluate_campaign_DEFAULT.ipynb`.

1. **Overview and consistency** of the measurements (same fundamental component, coil currents, harmonics decrease
   with Z).
2. **Equivalent-source method**: active two-terminal equivalent per harmonic with a common angle reference
   (U_L1-L2), normalization of the source fluctuation to the voltage level before/after the fault, two estimators in
   comparison, pairwise solution of two measurements, forecast error, jackknife and Monte Carlo uncertainty – and the
   **measured source** (U_q = pre-fault voltage of the faulted phase, Z_q from the tests), which carries beyond the
   tested additional impedances.
3. **Residual current at Z_E** with the measured source (or the fitted one, `SOURCE`), the other one for comparison,
   scaling to the quantile of a **PQ measurement** of the busbar (read from the export of the PQ instrument: 10-min
   values, largest phase, quantile, factor by the duration) and your limits of the residual current; **resistive
   approach** (after the VDE Impuls on the residual earth-fault current).
4. **Stations** – earth potential rise and touch voltage per frequency (direct and fault-related), comparison with
   the earthing measurements from **groundmeas**, **reduction factor** (per harmonic and RMS value) and – with PEN
   currents – current into the **LV network** and Z_E,LV with the parallel check.
5. **Report tables** and **figures** – export to `<results>/campaign` and `<figures>/campaign`.
6. **groundmeas** – enter all measurements of the campaign as staged fault tests (dry run, then on request).
7. **Report** as a Word file (`REPORT`) with placeholders for the photos on site – in `<report>`.

In [ ]:
%matplotlib inline
import json
import sys
from pathlib import Path

LANGUAGE = "en"                # output language of tables, hints and figures: "en" | "de"
try:
    import groundfault as gf
except ImportError:            # not installed: use the source folder of a clone (…/groundfault/src)
    for p in [Path.cwd(), *Path.cwd().parents]:    # ./src, ./groundfault/src, ./*/groundfault/src
        try:
            candidates = [p / "src", p / "groundfault" / "src", *sorted(p.glob("*/groundfault/src"))]
        except OSError:
            candidates = [p / "src", p / "groundfault" / "src"]
        hits = [c for c in candidates if (c / "groundfault" / "__init__.py").is_file()]
        if hits:
            sys.path.insert(0, str(hits[0]))
            break
    import groundfault as gf
import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from groundfault import (campaign, corrections, demo, diagnostics, disturbances, document, earthing,
                         equivalent_source, explorer, i18n, interactive, layout, legacy, pq, register, report, rms, sites,
                         transformer)

i18n.set_language(LANGUAGE)
# work folder = this folder or the first folder above it with a campaigns folder ("campaigns" or
# "01_Messkampagnen"); its name gives the folder layout, without one the layout of LANGUAGE is used
WORK_FOLDER, LAYOUT = layout.find_work_folder(Path.cwd(), default=LANGUAGE)
WORK_FOLDER = Path(WORK_FOLDER)
pd.set_option("display.max_columns", 60); pd.set_option("display.width", 220)
pd.set_option("display.float_format", lambda x: "%.4g" % x)
report.activate_style()        # notebook preview = typeface of the exported figures


def show(df):
    """Table with column names and codes in the output language."""
    display(i18n.localize(df))


print("groundfault %s · work folder: %s · layout: %s · font of the figures: %s"
      % (gf.__version__, WORK_FOLDER, LAYOUT.language, report.used_font()))

## 1 · Configuration

In [ ]:
CAMPAIGN = "2026-01-15_Demo"
DEMO     = True      # create the demo campaign and evaluate its recordings if the result files are missing
# Result files of evaluate_measurement.ipynb: <results>/<measurement>/<measurement>_result.json
# Order = columns of the report tables. Z = additional impedance in Ω (None → not in the equivalent-source method)
MEASUREMENTS = [
    dict(name="Measurement 1 (SUB 2.5 Ω)", measurement="SUB_2_5_10_000kHz", Z=2.5),
    dict(name="Measurement 2 (SUB 5 Ω)",   measurement="SUB_5_10_000kHz",   Z=5.0),
    dict(name="Measurement 3 (SUB 0 Ω)",   measurement="SUB_0_10_000kHz",   Z=0.0),
    dict(name="Measurement 4 (Station)",   measurement="Station_Elm_Road_10_000kHz", Z=None),
]
# optional per measurement: location="…" (exact location name in groundmeas, otherwise from the sites file or the
# site), description="…" (text before the automatic groundmeas description), timestamp="YYYY-MM-DD hh:mm:ss" (UTC,
# only for a wrongly set recorder clock)

# ---------------------------------------------------------------- equivalent-source method
ORDERS        = "auto"             # "auto" = odd orders with ≥ 0.5 A in ≥ 2 measurements, otherwise e.g. [3, 5, 7, 11, 13]
NORMALIZATION = "complex"          # "complex" (magnitude and phase) | "magnitude" | None
LEVEL         = "mean"             # source level: "pre_fault" | "post_fault" | "mean"
METHOD        = "equation_error"   # "equation_error" (linear least squares) | "output_error" (nonlinear)
SOURCE        = "measured"         # source of the result I_RES(Z_E): "measured" (pre-fault voltage) | "fitted" (from the currents)
SOURCE_IMPEDANCE = "median"        # Z_q of the measured source: "median" over the tests | "smallest" (test without additional impedance, the others check it)
Z_E           = 2.5                # demo value: earthing impedance in Ω at which I_RES is computed
Z_E_MAX       = 10.0               # end of the axis of the figure I_RES(Z_E)
IG_CHOICE     = "max"              # fundamental component: "max" (conservative) | "median" | a number in A

# ---------------------------------------------------------------- scaling to the 95 % quantile of a PQ measurement
# U95 scales every order of the positive or negative sequence (5, 7, 11, 13, 17 …) with its own 95 % value – also in
# the remainder outside the orders of the model; THD95_PERCENT scales flat with THD95/THD of the series the orders
# without their own value (alone: all of them). The zero sequence (3, 9, 15 …) and the even orders stay as measured.
U95 = None                         # per order in V, e.g. {5: U_5, 7: U_7}
U95_REFERENCE = "LL"               # "LL" = line-to-line (PQ device) | "seq" = sequence voltage (line-to-earth)
THD95_PERCENT = None               # flat: 95 % quantile of the THD in % (line-to-line), alone or together with U95
# With PQ_FILES section 4 takes U95 (line-to-earth, largest phase per 10-min interval) and THD95_PERCENT (largest loop)
# from the export(s) of a PQ instrument at the busbar – the values above are then replaced.
PQ_FILES      = ["raw/PQ/SUB_PQ_export.csv"]   # demo: PQ export of the demo busbar (written with the demo campaign)
PQ_TIMEZONE   = None               # time zone of the instrument clock, e.g. "Europe/Berlin" (None = time stamps as exported)
PQ_QUANTILE   = 0.95               # quantile of the 10-min values

# ---------------------------------------------------------------- limits, margins and factors – defined by you
# groundfault contains no limits or safety factors: take them from the operator's specification or the applicable
# standard. Not set (the values below), the parts that need them are skipped with a note.
LIMITS            = ()      # limits of the residual current in A, e.g. (I_1, I_2): lines in the figures I_RES(Z_E)
U_TP              = None    # permissible touch voltage U_Tp in V (resistive approach; reduction factor: None = from the result file)
F_FACTOR          = None    # factor F of the permissible earth potential rise F·U_Tp (as U_TP)
REFERENCE_CURRENT = None    # reference current in A for the reduction factor (None = largest limit of the result file)
UNCERTAINTY_I     = 0.0     # measurement-uncertainty margin m_I of currents for I_RES at Z_E: I·(1 + m_I) (0.0 = none)
UNCERTAINTY_U     = 0.0     # … m_U of voltages: an order scaled with a measured voltage takes k_m/(1 − m_U) (0.0 = none)
LOO_LIMIT_PCT     = None    # leave-one-out check: permissible deviation in % of every test from the model of the others (None = without assessment)
LOO_LIMIT_A       = None    # … and in A, for the small currents of the higher orders (within one of the two = passed; None = not used)
PQ_SAFETY_FACTOR  = 1.0     # factor on the 95 % quantiles of the PQ measurement (1.0 = none)
PQ_FACTORS        = None    # factor on the quantile by the duration of the PQ measurement {minimum days: factor} (None = PQ_SAFETY_FACTOR)
R_TYPICAL         = None    # range (low, high) of the reduction factor at 50 Hz you expect, e.g. from your experience

# ---------------------------------------------------------------- groundmeas (comparison section 5, entries section 7)
GROUNDMEAS_DB = str(WORK_FOLDER / "demo_groundmeas.db")   # demo database (written with the demo campaign)
GROUNDMEAS_EXPORT = False          # True: enter the measurements (one backup copy before); "replace": renew own entries
GROUNDMEAS_UNCERTAIN = "mark"      # uncertain fault-related values: "mark" (with a remark) | "omit"
GROUNDMEAS_UNITS = "suitable"      # "suitable": readable per quantity (mV, mA …; groundmeas ≥ 1.6.1) | "base": V and A

# ---------------------------------------------------------------- resistive approach (data of your network)
RESISTIVE = dict(U_thd95_V=None,   # harmonic voltage (95 %) in V (None = skip)
                 from_ll=True,     # True: the value is line-to-line → /√3
                 R_E_max=None, R1_km=None, R0_km=None,   # largest earthing resistance in Ω; Ω/km at the dominant frequency
                 r=1.0,            # reduction factor; r < 1: r·R_E acts in the fault circuit
                 I_res_max=None)   # permissible residual current in A (None = F_FACTOR·U_TP/(r·R_E_max))

EXPORT_FORMATS = ("pdf", "svg", "png")

# ---------------------------------------------------------------- report (Word, section 8)
# Title page and texts (keys of groundfault.document.ReportInfo; None = no report). Names of the network, places and
# persons appear only as you give them here; limits, U_Tp, F and margins are those of this configuration.
REPORT = dict(network="20 kV network of the demo substation",   # demo: invented names
              authors=["Demo author"],
              expansion=True,                             # network expansion (section 3.3) – e.g. dict(I_CE=800.0): the planned I_CE in A
              places={"SUB": "Demo substation", "Station_Elm_Road": "Station Elm Road"})
PHOTOS = {}                        # photos on site in <media>: {"setup_impedance": …, "recorder": …, "station": …} (missing: placeholder)
REPORT_TEMPLATE = None             # Word file of your organisation (styles, header and footer, logo), e.g. made with document.make_template
REPORT_STYLES = None               # style names of the template, e.g. {"title": "…", "subtitle": "…", "table": "…"}
REPORT_PDF = False                 # also write a PDF with LibreOffice (if installed)

## 2 · Load the results – overview and consistency

In [ ]:
CAMPAIGN_FOLDER = WORK_FOLDER / LAYOUT.campaigns / CAMPAIGN
for m in MEASUREMENTS:                 # known keys only (a misspelt or former key would be ignored)
    legacy.check_keys("MEASUREMENTS", m, ("name", "measurement", "Z", "location", "description", "timestamp"))


def result_file(measurement):
    """Result file of a measurement (also with the suffix of the other layout)."""
    p = Path(LAYOUT.result_path(CAMPAIGN_FOLDER, measurement))
    other = [p.parent / (measurement + lay.result_suffix) for lay in layout.LAYOUTS.values()]
    return next((x for x in [p, *other] if x.exists()), p)


if DEMO:                               # demo: create the campaign and evaluate the recordings with their known intervals
    if not CAMPAIGN_FOLDER.exists():
        demo.make_campaign(str(WORK_FOLDER), CAMPAIGN, layout=LAYOUT.language)
    CORR = corrections.load(CAMPAIGN_FOLDER)      # corrections of the campaign, as in evaluate_measurement.ipynb
    for m in MEASUREMENTS:
        if not result_file(m["measurement"]).exists():
            cfg = next(CAMPAIGN_FOLDER.glob("%s/*/%s.cfg" % (LAYOUT.raw, m["measurement"])))
            with open(str(cfg).replace(".cfg", "_truth.json"), encoding="utf-8") as f:
                b = json.load(f)["intervals"]
            rec_ = gf.open_recording(str(cfg))
            CORR.apply(rec_)
            roles_ = gf.detect_roles(rec_.channel_names, rec_.units, CORR.role_overrides(rec_), CORR.main_channel(rec_),
                                     CORR.distances(rec_))
            r_ = gf.evaluate(rec_, gf.Intervals(*(tuple(b[k]) for k in ("steady_state", "pre_fault", "post_fault"))),
                             gf.Settings(uncertainty_I=UNCERTAINTY_I, uncertainty_U=UNCERTAINTY_U, ires_limits=LIMITS,
                                         u_tp=U_TP, f_factor=F_FACTOR),
                             roles=roles_, z_fault_ohm=m["Z"])
            r_.save_json(LAYOUT.result_path(CAMPAIGN_FOLDER, m["measurement"]))
RESULTS = {}
for m in MEASUREMENTS:
    p = result_file(m["measurement"])
    if p.exists():
        RESULTS[m["name"]] = gf.load_result(str(p), m["name"])
    else:
        print("⚠ missing:", p.relative_to(WORK_FOLDER), "→ evaluate it with evaluate_measurement.ipynb first")
show(campaign.overview(RESULTS))
display(Markdown("**Consistency between the measurements**"))
show(campaign.consistency({m["name"]: RESULTS[m["name"]] for m in MEASUREMENTS if m["Z"] is not None and m["name"] in RESULTS}))
for n, e in RESULTS.items():
    for h in e.hints:
        print("⚠ %s: %s" % (n, h))

## 3 · Equivalent-source method (active two-terminal equivalent per harmonic)

**Normalization:** the harmonic level of the network fluctuates between the measurements (minutes apart). Each current
is referred to the level of the natural sequence measured at the same time *at the faulted phase* (5th/11th negative
sequence, 7th/13th positive sequence; for L2/L3 rotated by ±120°). Orders of the zero sequence (3rd, 9th) are **not**
normalized – their current only arises in the earth fault.
**Two estimators:** *equation error* (linear least squares) and *output error* (minimises the current deviation). If
they disagree strongly, U_q and Z_q are individually uncertain (the hint says whether the forecast within the measured
range is stable nevertheless).
**Thevenin check:** physically U_q is the pre-fault voltage of the faulted phase → column `Uq_ref_faulted_phase` ≈ 1 and
`Uq_angle_ref_faulted_phase_deg` ≈ 0°. A deviation of more than 10 % or 10° shows that the fitted U_q has absorbed
changes of the network between the tests: U_q is the slope of 1/I over Z, poorly determined when the additional
impedances are small compared with |Z_q|. Within the tested range the model current stays reliable, beyond it not.
**Sensitivity** `dZq_rel_per_1pct`: relative change of Z_q for an error of 1 % (or 0.5°) of one current – above 0.1
ill-conditioned.
**Measured source:** U_q as measured – the pre-fault voltage of the faulted phase – and Z_q = U_q/I_k − Z_k from the
tests (`SOURCE_IMPEDANCE`: the median over the tests or the test without additional impedance); one test suffices, every
further test is checked against the model of the others (leave-one-out, `LOO_LIMIT_PCT`). The table of the single
tests shows how much the network changed between them, the comparison where the two sources agree (tested range) and
where not. Orders of the zero sequence keep the fitted source – their source
arises in the fault. `SOURCE` selects the result of the checks and of section 4; the other source is drawn dashed.

In [ ]:
SERIES = MEASURED = MAIN = OTHER = MC = None   # fitted source, measured source, result (SOURCE), comparison; MC: Monte Carlo (checks)
SOURCE_TEXT = {"measured": "measured source", "fitted": "fitted source"}
entries = [dict(file=str(result_file(m["measurement"])), Z=m["Z"], name=m["name"])
           for m in MEASUREMENTS if m["Z"] is not None and m["name"] in RESULTS]
if entries:
    legacy.check_option("SOURCE", SOURCE, ("measured", "fitted"))
    legacy.check_option("SOURCE_IMPEDANCE", SOURCE_IMPEDANCE, ("median", "smallest"))
    TESTS = equivalent_source.load_series(entries)
    if len({complex(e["Z"]) for e in entries}) >= 2:
        SERIES = equivalent_source.fit_equivalent_source(TESTS, orders=ORDERS, normalization=NORMALIZATION,
                                                         reference=LEVEL, method=METHOD)
        MEASURED = equivalent_source.measured_source(SERIES, impedance=SOURCE_IMPEDANCE)
    else:                              # one additional impedance: no fitted source, the measured source suffices
        MEASURED = equivalent_source.measured_source(TESTS, impedance=SOURCE_IMPEDANCE, orders=ORDERS, reference=LEVEL)
        print("ℹ One additional impedance only: the fitted source needs two – result with the measured source.")
    MAIN, OTHER = (MEASURED, SERIES) if SOURCE == "measured" or SERIES is None else (SERIES, MEASURED)
    show(MAIN.measurements_table())
    display(Markdown("**Source level before/after the fault** (drift within one measurement = quality of the normalization)"))
    show(MAIN.level_table(LEVEL))
    if SERIES is not None:
        display(Markdown("**Fitted source: two-terminal equivalents per order** (Thevenin check ≈ 1∠0°)"))
        show(SERIES.source_table())
        for h in SERIES.hints:
            print("ℹ", h)
        for m, F in SERIES.fits.items():
            for h in F.hints:
                print("ℹ harmonic %d: %s" % (m, h))
    display(Markdown("**Measured source** – U_q = pre-fault voltage of the faulted phase; Z_q of every single test (the spread is the change of the network between the tests) and of the series"))
    show(MEASURED.single_test_table())
    show(MEASURED.source_table()[["order", "f_Hz", "sequence", "method", "Uq_V", "Rq_Ohm", "Xq_Ohm", "Zq_Ohm",
                                  "residual_pct", "Rq_fixed"]])
    for h in (MEASURED.hints[-1:] if SERIES is not None else MEASURED.hints):    # with a fitted series: only the new hint
        print("ℹ", h)
    for m, F in MEASURED.fits.items():
        for h in (F.hints if F.method == "measured" else F.hints[-1:]):     # kept fitted orders: only the new hint
            print("ℹ harmonic %d: %s" % (m, h))
    if SERIES is not None:
        display(Markdown("**Measured against fitted source** – current per order and I_RES within and beyond the tested range (`fitted_to_measured`; `interpolation` = within the tested additional impedances)"))
        zmin, zmax = SERIES.Z_range
        show(equivalent_source.compare_sources(MEASURED, SERIES, [0.0, zmax, 2 * zmax]
                                               + ([Z_E] if Z_E is not None and Z_E not in (0.0, zmax, 2 * zmax) else []),
                                               IG=IG_CHOICE))
else:
    print("The equivalent-source method needs at least one substation measurement with its additional impedance Z (the fitted source two different ones).")

### Checks: pairwise solution, forecast per measurement, jackknife, Monte Carlo

In [ ]:
if MAIN is not None:
    if SERIES is not None:
        display(Markdown("**Fitted source: solution of every pair of tests**"))
        show(SERIES.pairwise_table())
    display(Markdown(i18n.format_text("**Result ({}): model against measurement per test, jackknife at Z_E**",
                                      SOURCE_TEXT[MAIN.settings["source"]])))
    show(MAIN.forecast_table())
    display(Markdown("**Leave-one-out: every test from the model of the other tests** (a large deviation: the network or the sources changed between the tests)"))
    LOO = MAIN.loo_table(LOO_LIMIT_PCT, LOO_LIMIT_A)
    show(LOO)
    BAD = LOO["within_limit"].eq(False) if "within_limit" in LOO else LOO["order"].isna()   # None: not determinable
    if BAD.any():
        print(i18n.format_text("⚠ Leave-one-out: {} of {} values outside the permissible deviation ({}) – a hint that the network or the sources changed between the tests",
                               int(BAD.sum()), int(LOO["within_limit"].notna().sum()),
                               " / ".join(i18n.format_text(f, v) for f, v in (("±{:g} %", LOO_LIMIT_PCT), ("±{:g} A", LOO_LIMIT_A)) if v is not None)))
    if Z_E is not None:
        show(MAIN.jackknife_table(Z_E))
    display(Markdown("**Residual current at the measured impedances** – model (with the level and I_G of each measurement, rest = orders not fitted) against the measured true RMS value"))
    show(MAIN.check_table())
    if Z_E is None:
        print("ℹ Jackknife and Monte Carlo at Z_E skipped – Z_E is not set (section 1).")
    else:
        MC = equivalent_source.monte_carlo(MAIN, Z_E, n=2000, IG=IG_CHOICE)
        print(i18n.format_text("Monte Carlo (1 %/0.5° current, 2 % Z, model residual): I_RES(Z_E = {:g} Ω) P5 … P95 = {:.1f} … {:.1f} A (median {:.1f} A)",
                               Z_E, MC["IRES"][5], MC["IRES"][95], MC["IRES"][50]))
        zmin, zmax = MAIN.Z_range
        print(i18n.format_text("Z_E lies {} the measured range {:.1f} … {:.1f} Ω",
            "within" if zmin <= abs(Z_E) <= zmax else "OUTSIDE (extrapolation)", zmin, zmax))
    interactive.source_fit(MAIN, Z_E, comparison=OTHER).show()

## 4 · Residual current at the earthing impedance and resistive approach

### Power-quality measurement of the busbar (PQ)

With `PQ_FILES` the export(s) of a PQ instrument at the busbar replace `U95`, `U95_REFERENCE`, `THD95_PERCENT` and
`PQ_SAFETY_FACTOR` of section 1: 10-min values (from shorter intervals as the square root of the mean of the
squares), per interval the **largest phase** (harmonic voltage line-to-earth per order) or **loop** (THD
line-to-line), of these values the quantile `PQ_QUANTILE`. The factor on the quantile follows from the longest stretch
of **gapless** values and `PQ_FACTORS`; shorter than every duration of `PQ_FACTORS` the measurement is not valid and
nothing is scaled with it. The PQ measurement and the tests must refer to the same network and switching state.
Without a value in `RESISTIVE["U_thd95_V"]` the resistive approach gets the quantile of the harmonic voltage
(line-to-line) times the factor. The table of the tests shows where their pre-fault voltages lie in the distribution
of the PQ measurement: a low percentile is normal for a test at a time of low harmonic voltage (night, weekend) –
compare the time window of the same weekday; a test level above the largest value of the PQ measurement points to a
different network state (switching state, capacitances, transformer). Figures: 10-min values over the period, duration
curve with the levels of the tests and the weekly profile.

In [ ]:
PQ = PQ_COMPARISON = PQ_LEVELS = None
if PQ_FILES:
    files = [CAMPAIGN_FOLDER / p for p in ([PQ_FILES] if isinstance(PQ_FILES, (str, Path)) else PQ_FILES)]
    PQ = pq.assess(pq.read(files, timezone=PQ_TIMEZONE), quantile=PQ_QUANTILE,
                   **(dict(factors=PQ_FACTORS) if PQ_FACTORS else dict(factor=PQ_SAFETY_FACTOR)))
    print(PQ.summary())
    for h in PQ.hints:
        print("ℹ", h)
    show(PQ.table().query("order >= 2"))
    show(PQ.summary_table())
    if PQ.valid:                       # the values of the measurement replace those of section 1
        U95, U95_REFERENCE = PQ.u95(), ("seq" if PQ.reference == "LE" else "LL")
        THD95_PERCENT, PQ_SAFETY_FACTOR = (PQ.thd95() if np.isfinite(PQ.thd95()) else None), PQ.factor
        if RESISTIVE.get("U_thd95_V") is None and np.isfinite(PQ.uh95_ll):
            RESISTIVE = dict(RESISTIVE, U_thd95_V=PQ.uh95_ll * PQ.factor, from_ll=True)
            print(i18n.format_text("Resistive approach: U_thd95_V = {:.1f} V (quantile of the harmonic voltage line-to-line × {:g})",
                                   RESISTIVE["U_thd95_V"], PQ.factor))
    else:
        print("⚠ The PQ measurement is not valid for the assessment (gapless values shorter than every duration of PQ_FACTORS) – nothing is scaled with it; U95/THD95_PERCENT of section 1 stay.")
    PQ_LEVELS, times = {}, {}
    for m in MEASUREMENTS:
        if m["Z"] is not None and m["name"] in RESULTS:
            PQ_LEVELS[m["name"]], times[m["name"]] = pq.pre_fault_levels(RESULTS[m["name"]], kind=PQ.reference)
    orders = sorted(o for o in (MAIN.fits if MAIN is not None else PQ.u95()) if gf.formulas.harmonic_category(o) == "pos_neg"
                    and o in PQ.data.orders)
    PQ_COMPARISON = PQ.comparability(PQ_LEVELS, times, orders=orders or None)
    for h in PQ_COMPARISON.attrs.get("hints", []):
        print("ℹ", h)
    display(Markdown("**Fault tests in the distribution of the PQ measurement** – share of the 10-min values below the pre-fault voltage of the test (largest phase) over the whole measurement and in its time window; `factor_to_test` = quantile/level of the test"))
    show(PQ_COMPARISON)
    report.pq_series(PQ)
    report.pq_duration(PQ, levels={n: lv[PQ.dominant_order] for n, lv in PQ_LEVELS.items() if PQ.dominant_order in lv})
    report.pq_profile(PQ)
else:
    print("ℹ No PQ measurement (PQ_FILES) – U95/THD95_PERCENT of section 1 are used if set.")

In [ ]:
SCALING = None
if MAIN is not None:
    if U95 or THD95_PERCENT:           # U95 per order; THD95_PERCENT for the orders without their own value (alone: all of them)
        SCALING = dict(safety_factor=PQ_SAFETY_FACTOR)
    if U95:
        SCALING.update(u95=U95, u95_reference=U95_REFERENCE)
    if THD95_PERCENT:
        thd_measured = float(pd.concat([e.tables["voltage_metrics"] for n, e in RESULTS.items()
                                        if n in [x["name"] for x in entries]])
                             .query("interval != 'fault'")[["UTHD12_pct", "UTHD23_pct", "UTHD31_pct"]].max(axis=1).mean())
        SCALING.update(thd95_percent=THD95_PERCENT, thd_measured_percent=thd_measured)
        print(i18n.format_text("THD of the series (mean before/after, largest loop): {:.2f} %", thd_measured))
    if Z_E is None:
        print("ℹ Residual current at Z_E skipped – Z_E is not set (section 1); the figure shows I_RES over Z_E.")
    else:
        r = equivalent_source.ires_at_impedance(MAIN, Z_E, IG=IG_CHOICE, uncertainty_I=UNCERTAINTY_I,
                                                uncertainty_U=UNCERTAINTY_U, **(SCALING or {}))
        margin = UNCERTAINTY_I > 0 or UNCERTAINTY_U > 0   # the value with the margin only if you set one
        print(i18n.format_text("{}: I_G = {:.2f} A | I_H(Z_E) = {:.2f} A (rest {:.2f} A) | I_RES(Z_E = {:g} Ω) = {:.2f} A",
                               SOURCE_TEXT[MAIN.settings["source"]], r["IG"], r["IH"], r["IH_rest"], Z_E, r["IRES"])
              + (i18n.format_text(" | with the measurement-uncertainty margin: {:.2f} A", r["IRES_assessed"]) if margin else ""))
        if SCALING:
            print(i18n.format_text("scaled: I_H = {:.2f} A → I_RES = {:.2f} A", r["IH_scaled"], r["IRES_scaled"])
                  + (i18n.format_text(" (with the margin: {:.2f} A)", r["IRES_scaled_assessed"]) if margin else "")
                  + i18n.format_text(" | not scaled: zero sequence {:.2f} A, even orders {:.2f} A, without an order {:.2f} A",
                                     r["IH_zero"], r["IH_even"], r["IH_other"]))
        display(Markdown(i18n.format_text("**Result per order at Z_E = {:g} Ω** – the orders of the model at Z_E, the remainder of the test {} (the largest one); rule: `own voltage` (positive/negative sequence, quantile/level of the tests × factor), `THD` (without their own value), `as measured` (zero sequence, even orders)",
                                          Z_E, r["rest_test"] or "–")))
        show(r["table"])
        for h in r["hints"]:
            print("ℹ", h)
        if OTHER is not None:
            o = equivalent_source.ires_at_impedance(OTHER, Z_E, IG=IG_CHOICE, **(SCALING or {}))
            print(i18n.format_text("comparison {}: I_H(Z_E) = {:.2f} A | I_RES(Z_E) = {:.2f} A",
                                   SOURCE_TEXT[OTHER.settings["source"]], o["IH"], o["IRES"])
                  + (i18n.format_text(" | scaled: I_RES = {:.2f} A", o["IRES_scaled"]) if SCALING else ""))
    if not LIMITS:
        print("ℹ No limit lines in the figures I_RES(Z_E) – LIMITS is not set (section 1).")
    interactive.ires_ze(MAIN, Z_E_MAX, LIMITS, SCALING, comparison=OTHER).show()
legacy.check_keys("RESISTIVE", RESISTIVE, ("U_thd95_V", "from_ll", "R_E_max", "R1_km", "R0_km", "r", "I_res_max"))
NOT_SET = [k for k in ("U_thd95_V", "R_E_max", "R1_km", "R0_km") if RESISTIVE.get(k) is None]
if RESISTIVE.get("I_res_max") is None:                # otherwise I_RES,max = F·U_Tp/(r·R_E,max)
    NOT_SET += [n for n, v in (("U_TP", U_TP), ("F_FACTOR", F_FACTOR)) if v is None]
if NOT_SET:
    print("ℹ Resistive approach skipped – not set (section 1): %s"
          % ", ".join(n if n in ("U_TP", "F_FACTOR") else "RESISTIVE[%r]" % n for n in NOT_SET))
else:
    IG_ALL = [e.values["IG"] for e in RESULTS.values() if np.isfinite(e.values.get("IG", np.nan))] or [0.0]
    if isinstance(IG_CHOICE, str):
        legacy.check_option("IG_CHOICE", IG_CHOICE, ("max", "median"))
    IG = (max(IG_ALL) if IG_CHOICE == "max" else float(np.median(IG_ALL)) if IG_CHOICE == "median" else float(IG_CHOICE))
    display(pd.Series(campaign.resistive_approach(RESISTIVE["U_thd95_V"], IG, RESISTIVE["R_E_max"], RESISTIVE["R1_km"],
                                                  RESISTIVE["R0_km"], I_res_max=RESISTIVE.get("I_res_max"),
                                                  U_Tp=U_TP, F=F_FACTOR, from_ll=RESISTIVE.get("from_ll", False),
                                                  r=RESISTIVE.get("r", 1.0)),
                      name="resistive approach"))

### PQ scaling at Z_E: per order, as a time series and with the THD

**Result** (`SCALING`): every order of the positive or negative sequence (5, 7, 11, 13, 17 …) with its own factor
quantile/level of the tests times the factor – also in the remainder outside the orders of the model; an order without
its own value (no quantile, level below 1 V) with the THD factor. The zero sequence (3, 9, 15 …) arises in the earth
fault and stays as measured; the even orders stay as measured and are listed separately (atypical – a hint names them
from 5 % of the harmonic content). **Time series**: the same for every 10-min value, of these I_RES the quantile
`PQ_QUANTILE` (times the factor) – smaller when the orders do not reach their high values at the same time. **THD**:
one flat factor THD quantile/THD of the tests for every order of the positive or negative sequence – the rule without
values per order.

In [ ]:
PQ_SCALING = None
if PQ is not None and PQ.valid and MAIN is not None and Z_E is not None and SCALING:
    PQ_SCALING = PQ.scaling_methods(MAIN, Z_E, IG=IG_CHOICE, uncertainty_I=UNCERTAINTY_I, uncertainty_U=UNCERTAINTY_U,
                                    scaling=SCALING)
    display(Markdown(i18n.format_text("**Residual current at Z_E = {:g} Ω with the PQ measurement** (quantile {:g}, factor {:g}; assessed = with UNCERTAINTY_I and UNCERTAINTY_U)",
                                      Z_E, PQ_QUANTILE, PQ.factor)))
    show(PQ_SCALING)
    for h in PQ_SCALING.attrs.get("hints", []):
        print("ℹ", h)
    for limit in LIMITS:
        over = PQ_SCALING.loc[PQ_SCALING["IRES_assessed_A"] > limit, "method"]
        print(i18n.format_text("I_RES above {:g} A: ", limit) + (", ".join(over) or "–"))
elif PQ is not None and not PQ.valid:
    print("ℹ No PQ scaling at Z_E – the PQ measurement is not valid for the assessment (section 4, PQ).")
elif PQ is not None:
    print("ℹ No comparison at Z_E – Z_E or the equivalent sources are missing (sections 1 and 3).")

## 5 · Stations: earth potential rise and touch voltage per frequency, earthing measurement (groundmeas)

For every station measurement of the campaign (result file with the table `station_frequency`): voltage per ampere of
fault current over the frequency – **direct** as measured and **fault-related** (without the background before the
fault) – and, if the location is in groundmeas, the comparison with the earthing measurement (injection) with the
ratio (for the earth potential rise at a remote probe = effective reduction factor). groundmeas is only read.
**Reduction factor** per harmonic and as RMS value (earth potential rise per ampere, fault test ÷ earthing measurement,
magnitudes; decisive is the probe maximum `max(UE)` against the farthest probe of the earthing measurement) with
classification (against your range `R_TYPICAL` at 50 Hz, falling with f) and relevance of the deviation for the
assessment (voltage at the reference current against F·U_Tp – `REFERENCE_CURRENT`, `U_TP` and `F_FACTOR` of section 1,
otherwise the values stored in the result file); with PEN currents also the current into the **LV network** and
Z_E,LV = ΔU_E/I_LV with the parallel check (against the fault test max(U_E)/I_F and the earth current) and the
comparison with the earthing measurement.

In [ ]:
STATIONS = {n: e for n, e in RESULTS.items() if not e.tables.get("station_frequency", pd.DataFrame()).empty}
if STATIONS and R_TYPICAL is None:
    print("ℹ No classification of r at 50 Hz – R_TYPICAL is not set (section 1).")
COMPARISONS = {}
REDUCTION = {}                                                    # reduction factor and parallel check per station
DB = None
try:
    SITES = sites.load(CAMPAIGN_FOLDER)                            # sites file (location in groundmeas)
except ValueError as ex:
    SITES = sites.Sites()
    print("⚠", ex)
if STATIONS:
    try:
        DB = earthing.Database(GROUNDMEAS_DB)
    except (FileNotFoundError, ValueError) as ex:
        print("ℹ groundmeas database not available:", ex)
else:
    print("No station measurement with an evaluation per frequency.")
for n, e in STATIONS.items():
    display(Markdown("### %s" % n))
    show(e.table("station_frequency"))
    interactive.station_frequency(e).show()
    if not e.tables.get("station_lv_network", pd.DataFrame()).empty:
        display(Markdown("**LV network** – current I_LV and Z_E,LV = ΔU_E/I_LV (per frequency and RMS value)"))
        show(e.table("station_lv_network_eff")); show(e.table("station_lv_network"))
    if DB is None:
        continue
    m = next(x for x in MEASUREMENTS if x["name"] == n)
    site = e.data.get("meta", {}).get("site") or ""
    entry = SITES.for_site(site)
    location = m.get("location") or (entry.location if entry is not None and entry.location else None) or \
        earthing.location_from_site(site or m["measurement"])
    choice = DB.choose_location(location, exact_only=bool(m.get("location") or (entry is not None and entry.location)),
                                preset='location="{}" in MEASUREMENTS')
    if choice["note"]:
        print("ℹ", choice["note"])
    if choice["location"] is None:
        continue
    meas = earthing.exclude_own(DB.measurements(location=choice["location"]), CAMPAIGN, m["measurement"])
    response = earthing.frequency_response(DB, meas["measurement_id"])
    long_df, ratio = earthing.compare(e.table("station_frequency"), response, measurement=n)
    COMPARISONS[n] = (long_df, ratio)
    print("groundmeas: location “%s”, measurements %s" % (choice["location"], ", ".join(map(str, meas["measurement_id"]))))
    if len(ratio):
        show(ratio)
    if long_df["source"].isin([earthing.SOURCE_INJECTION, earthing.SOURCE_GROUNDMEAS_FAULT]).any():
        interactive.earthing_comparison(long_df, ratio).show()
    rf, rf_eff = earthing.reduction_factor(e, response, reference_current_A=REFERENCE_CURRENT, U_Tp=U_TP, F=F_FACTOR,
                                           r_typical=R_TYPICAL)  # None: from the result file
    not_set = [v for v, c in (("REFERENCE_CURRENT", "reference_current_A"), ("U_TP", "U_Tp_V"), ("F_FACTOR", "F_factor"))
               if len(rf_eff) and rf_eff[c].isna().all()]
    if not_set:
        print("ℹ %s: relevance of the deviation for the assessment not determined – neither set in section 1 nor stored in the result file: %s"
              % (n, ", ".join(not_set)))
    lvc, lvc_eff = earthing.check_lv_network(e, response)
    REDUCTION[n] = dict(reduction_factor=rf, reduction_factor_eff=rf_eff, lv_network_check=lvc,
                        lv_network_check_eff=lvc_eff)
    if len(rf_eff):
        display(Markdown("**Reduction factor** (RMS value and per harmonic)")); show(rf_eff); show(rf)
        interactive.reduction_factor(rf, rf_eff).show()
    if len(lvc):
        display(Markdown("**LV network ↔ earthing measurement**")); show(lvc_eff); show(lvc)
        interactive.lv_network(e.table("station_lv_network"), lvc).show()
    for h in earthing.reduction_hints(rf_eff, lvc):
        print("⚠ %s: %s" % (n, h))

## 6 · Export the report tables and figures

Report tables (voltages before/after, residual current, split at 50 Hz, earth potential rises, shield current,
transformer, assessment quantities, equivalent sources) as an Excel workbook, a Word file, CSV and Markdown; figures
in final size without titles as PDF/SVG/PNG.

In [ ]:
slug = report.slug                  # names in file and table names: lower case, letters/digits/_ (as in the report tables)
OUT = CAMPAIGN_FOLDER / LAYOUT.results / ("campaign" if LAYOUT.language == "en" else "Kampagne")
FIG = CAMPAIGN_FOLDER / LAYOUT.figures / ("campaign" if LAYOUT.language == "en" else "Kampagne")
TABLES = report.report_tables(RESULTS, series=MAIN, Z_E=Z_E, comparison=OTHER)
TABLES["overview"] = campaign.overview(RESULTS)
for n, e in STATIONS.items():                     # station: per frequency and ratio to the earthing measurement
    TABLES["station_frequency_" + slug(n)] = e.table("station_frequency")
    if n in COMPARISONS and len(COMPARISONS[n][1]):
        TABLES["earthing_ratio_" + slug(n)] = COMPARISONS[n][1]
    if not e.tables.get("station_lv_network_eff", pd.DataFrame()).empty:
        TABLES["station_lv_network_eff_" + slug(n)] = e.table("station_lv_network_eff")
        TABLES["station_lv_network_" + slug(n)] = e.table("station_lv_network")
    for t_, df in REDUCTION.get(n, {}).items():
        if len(df):
            TABLES[t_ + "_" + slug(n)] = df
if PQ is not None:                                 # PQ measurement: quantiles, characteristic values, tests, residual current at Z_E
    TABLES["pq_quantiles"] = PQ.table().query("order >= 2")
    TABLES["pq_summary"] = PQ.summary_table()
    for name_, df_ in (("pq_tests", PQ_COMPARISON), ("pq_scaling", PQ_SCALING)):
        if df_ is not None and len(df_):
            TABLES[name_] = df_
for name, df in TABLES.items():                    # title with the name of the measurement as given
    display(Markdown("**%s**" % report.table_title(name, RESULTS))); display(report.format_table(df))
paths = report.export_tables(TABLES, str(OUT), prefix="campaign_", formats=("csv", "md"), names=RESULTS)
figures = []
for n, e in STATIONS.items():
    figures += report.save(report.station_frequency(e), str(FIG / ("station_frequency_" + slug(n))), EXPORT_FORMATS, close=True)
    figures += report.save(report.station_background(e), str(FIG / ("station_background_" + slug(n))), EXPORT_FORMATS, close=True)
    if n in COMPARISONS:
        try:
            figures += report.save(report.earthing_comparison(*COMPARISONS[n]),
                                   str(FIG / ("earthing_comparison_" + slug(n))), EXPORT_FORMATS, close=True)
        except ValueError as ex:
            print("ℹ %s: %s" % (n, ex))
    rd = REDUCTION.get(n, {})
    if len(rd.get("reduction_factor_eff", [])):
        figures += report.save(report.reduction_factor(rd["reduction_factor"], rd["reduction_factor_eff"]),
                               str(FIG / ("reduction_factor_" + slug(n))), EXPORT_FORMATS, close=True)
    if not e.tables.get("station_lv_network", pd.DataFrame()).empty:
        lvc = rd.get("lv_network_check")
        figures += report.save(report.lv_network(e.table("station_lv_network"), lvc if lvc is not None and len(lvc) else None),
                               str(FIG / ("lv_network_" + slug(n))), EXPORT_FORMATS, close=True)
sub_results = [RESULTS[m["name"]] for m in MEASUREMENTS if m["Z"] is not None and m["name"] in RESULTS]
if sub_results:
    f = report.spectrum_comparison(sub_results, [m["name"] for m in MEASUREMENTS if m["Z"] is not None and m["name"] in RESULTS])
    figures += report.save(f, str(FIG / "fault_current_spectrum_comparison"), EXPORT_FORMATS, close=True)
if MAIN is not None:                    # result (SOURCE), the other source dashed
    figures += report.save(report.source_fit(MAIN, Z_E=Z_E, comparison=OTHER), str(FIG / "source_fit"), EXPORT_FORMATS,
                           close=True)
    figures += report.save(report.source_plane(MAIN, MC), str(FIG / "source_impedance"), EXPORT_FORMATS, close=True)
    figures += report.save(report.ires_over_ze(MAIN, Z_E_MAX, LIMITS, IG_CHOICE, SCALING, comparison=OTHER),
                           str(FIG / "ires_over_ZE"), EXPORT_FORMATS, close=True)
if PQ is not None:
    figures += report.save(report.pq_series(PQ), str(FIG / "pq_series"), EXPORT_FORMATS, close=True)
    figures += report.save(report.pq_duration(PQ, levels={n: lv[PQ.dominant_order] for n, lv in (PQ_LEVELS or {}).items()
                                                          if PQ.dominant_order in lv}),
                           str(FIG / "pq_duration"), EXPORT_FORMATS, close=True)
    figures += report.save(report.pq_profile(PQ), str(FIG / "pq_profile"), EXPORT_FORMATS, close=True)
print("%d table files in %s" % (len(paths), OUT.relative_to(WORK_FOLDER)))
print("%d figure files in %s" % (len(figures), FIG.relative_to(WORK_FOLDER)))

## 7 · groundmeas: enter the measurements of the campaign

Every measurement of `MEASUREMENTS` as a staged fault test (`staged_fault_test`) in groundmeas – substation with the
fault current per order and the fault resistance, station also with the earth potential rise and touch voltage per
probe and frequency (fault-related part) and the shield current. Location, coordinates, asset type, voltage level and
distance to the substation come from the sites file of the campaign (created pre-filled from groundmeas the first
time – **enter coordinates for new locations**). At first only the overview (dry run); written with
`GROUNDMEAS_EXPORT = True` – after **one** backup copy of the database; measurements entered before are recognised and
skipped (`"replace"` renews the own entries), other entries are never changed.

In [ ]:
legacy.check_option("GROUNDMEAS_EXPORT", GROUNDMEAS_EXPORT, (False, True, "replace"))
PLANS, BACKUP = {}, True
for m in MEASUREMENTS:
    if m["name"] not in RESULTS:
        continue
    try:
        PLANS[m["name"]] = p = earthing.export_to_groundmeas(
            RESULTS[m["name"]], CAMPAIGN_FOLDER, GROUNDMEAS_DB, location=m.get("location"), write=bool(GROUNDMEAS_EXPORT),
            existing="replace" if GROUNDMEAS_EXPORT == "replace" else "skip", uncertain_mode=GROUNDMEAS_UNCERTAIN,
            description=m.get("description", ""), timestamp=m.get("timestamp"), units=GROUNDMEAS_UNITS, backup=BACKUP)
        if p.get("written"):
            BACKUP = p["backup"]           # one backup copy for all entries of this run
    except FileNotFoundError as ex:
        print("ℹ groundmeas database not available:", ex)
        break
    except ValueError as ex:
        print("ℹ %s: %s" % (m["name"], ex))
if PLANS:
    show(earthing.plan_overview(PLANS))
    for n, p in PLANS.items():
        for h in p["hints"]:
            print("ℹ %s: %s" % (n, h))
        for w in p["warnings"]:
            print("⚠ %s: %s" % (n, w))
    display(Markdown("**%s**" % Path(sites.path_for(CAMPAIGN_FOLDER)).relative_to(CAMPAIGN_FOLDER)))
    show(sites.load(CAMPAIGN_FOLDER).table())
    if isinstance(BACKUP, str):
        print("✓ entered – backup copy: %s – now set GROUNDMEAS_EXPORT back to False." % BACKUP)
    elif any(p["writable"] for p in PLANS.values()):
        print("Dry run – enter with GROUNDMEAS_EXPORT = True (values per measurement: earthing.plan_items(PLANS[name])).")

## 8 · Report (Word)

The report of the campaign as a Word file in `<report>`: title page, key results, contents, chapters 1 to 3 (setup
and procedure; results; assessment with the equivalent sources, the touch voltage at the stations, the uncertainties
and the summary) and appendices (data of the recordings, plausibility checks, symbols). Every number comes from the
objects of this notebook and the result files, waveforms and fault times from the raw recordings. Photos on site
(`PHOTOS`, file names in the media folder) are inserted, missing ones are framed placeholders; the recommendation stays
a highlighted placeholder until `REPORT["recommendation"]` gives it. With `REPORT_TEMPLATE` the report takes the
styles, header, footer and logo of your organisation. Open the file in Word and update the fields (contents, numbers of
figures and tables). Running this section again replaces an unchanged report; one saved in Word since is kept as
`<name>_<date>_<time>.docx`.

In [ ]:
REPORT_FILE = None
if REPORT is None:
    print("ℹ No report – REPORT is None (section 1).")
elif not document.docx_available():
    print("ℹ No report – python-docx is missing: pip install 'groundfault[export]'")
elif not RESULTS:
    print("ℹ No report – no results (section 2).")
else:
    REPORT_OUT = document.campaign_report(
        RESULTS, info=REPORT, measurements=MEASUREMENTS, campaign_folder=str(CAMPAIGN_FOLDER), series=MAIN,
        comparison=OTHER, monte_carlo=MC, loo=LOO if MAIN is not None else None, Z_E=Z_E, IG=IG_CHOICE, limits=LIMITS,
        u_tp=U_TP, f_factor=F_FACTOR, uncertainty_I=UNCERTAINTY_I, uncertainty_U=UNCERTAINTY_U, scaling=SCALING, pq=PQ,
        pq_levels=PQ_LEVELS, pq_comparison=PQ_COMPARISON, earthing=COMPARISONS, reduction=REDUCTION, photos=PHOTOS,
        template=REPORT_TEMPLATE, styles=REPORT_STYLES, language=LANGUAGE, pdf=REPORT_PDF)
    REPORT_FILE = Path(REPORT_OUT["path"])
    print("✓ report: %s%s" % (REPORT_FILE.relative_to(WORK_FOLDER), " + PDF" if REPORT_OUT["pdf"] else ""))
    print(i18n.format_text("  assessment with {} · {} figures", REPORT_OUT["basis"],
                           len(REPORT_OUT["figures"])))
    for p_ in REPORT_OUT["placeholders"]:     # still to fill in Word
        print("  [ ]", p_)
    for h in REPORT_OUT["hints"]:
        print("ℹ", h)